# HarnessChandelier

*Harness the Chandelier — connect to the center of your conversation.*  

**Topic Drift Tracking for Long-Running Agent Conversations**

In real-world AI agent interactions, users naturally drift across multiple topics —
then return to what they originally wanted.

HarnessChandelier doesn't try to prevent drift.
It **tracks it** — and finds the topic the user kept coming back to.

Like a chandelier at the center of a hall, the dominant topic stays fixed
no matter how much the conversation moves around it.

Built on:
- **BERTopic** + cuML UMAP/HDBSCAN — GPU-accelerated topic extraction
- **cuGraph PageRank** — topic importance ranking
- **Temporal edge weighting** — time-aware topic transition graph

Built for **Harness Engineering** workflows where conversations drift across topics.

This example demonstrates a scenario where an AI assistant repeatedly fails to follow the user's original design specifications, causing the user to continuously restate their core requirements throughout the conversation.

In [1]:
from harness_chandelier import HarnessChandelier

messages = [
    "Hey, I have this idea for a website. I want to build something like Figma but for automatically designing web pages. It should be blue tone based, not green, using Node.js on the backend and React for the frontend.",
    "The sidebar needs to be on the left side, and at the bottom a ChatGPT-style input box where I can type and also attach files.",
    "So to clarify, it's definitely blue tones only. Not green, not purple. Blue.",
    "When a person inputs a description, the site should automatically generate the whole web design for them. Like Figma but automated.",
    "I just tried running the code and it threw an error. What does this error message even mean?",
    "The input box is in the middle of the page right now. I want it moved down to the bottom like ChatGPT.",
    "I ran the code again and it's still showing an error. What is this message trying to tell me?",
    "Don't you remember? I said the website should be like Figma. You keep forgetting the main point.",
    "Ah another error popped up again. Do I really have to run it one more time?",
    "It's blue! I said blue website! Not the whole page, just the top part should be blue tones.",
    "About file uploads - if someone attaches a Python file or anything random, that might cause problems. Only allow image files.",
    "Like Figma, you input the design description and it should immediately output the actual designed web page.",
    "The layout should be centered, with the main content area in white but only the top part in blue tones.",
    "Still getting an error. This is the third time. Can you just fix it properly?",
    "I didn't even mention icon generation yet - what exactly does that part do in your implementation?",
    "The input box is too wide now. It needs to be properly centered and a reasonable width.",
    "Why did you change the blue color? I specifically said blue tones, not that shade.",
    "You know what, let me remind you again - this is supposed to work like Figma. Type description, get design. That's it.",
    "Another error. I don't understand why this keeps happening.",
    "The sidebar on the left is fine but I need it to be collapsible. Did you add that?",
    "Can we go back to basics? Blue tones on top, white content area, sidebar left, input box bottom. Simple.",
    "The error is different this time. It says something about Node.js modules not found.",
    "I want the generated designs to look professional, not like a template. More like what Figma produces.",
    "Stop changing things I didn't ask you to change. Just fix what I asked.",
    "The file upload button should only accept PNG and JPG files. Nothing else.",
    "Again with the errors! This code quality is really frustrating me.",
    "Do you remember the Figma-like functionality I mentioned at the very beginning? That's still the core feature.",
    "The blue color on top should be a gradient from dark blue to medium blue. Not flat.",
    "Wait, now the input box disappeared completely. What happened?",
    "I ran a fresh install and still getting dependency errors. This is taking too long.",
    "The auto-generated designs need to include proper typography. Not just layout blocks.",
    "I told you already - the input box goes at the BOTTOM of the page. Not the middle. Bottom.",
    "Can the sidebar show a history of previously generated designs? That would be useful.",
    "Yet another error. At this point I wonder if the code works at all.",
    "The whole point of this website is the Figma-like auto generation. Everything else is secondary.",
    "Now the blue styling broke on mobile. It looks completely different on my phone.",
    "Fix the error first, then we can talk about adding features.",
    "I need the generated web pages to be downloadable as HTML files.",
    "The top section blue color - it should match the shade #1E3A8A specifically.",
    "Still broken. The error message says 'Cannot read properties of undefined'.",
    "Remember, Node.js backend, React frontend. Don't switch to anything else.",
    "The ChatGPT-style input should also support voice input eventually, but that's for later.",
    "Why is the sidebar showing on the right now? I said LEFT sidebar.",
    "I just want it to work like I described from the start. Blue, Figma-like, sidebar left, input bottom.",
    "Error again. Same one as before. Did you not fix it properly the first time?",
    "The design generator needs to understand color schemes, not just layout.",
    "I'm starting to think you don't remember anything I said at the beginning of our conversation.",
    "Fix the Node.js error, make the top blue, put the input at the bottom. Three things. That's all.",
    "Like Figma. Auto design generation. From text description. Blue theme. Is that clear enough?",
    "If we can get the basic version working first, then we can add the icon generation feature later.",
    "Perhaps the first version you nade was imperferct but it was closer to what I wanted than this current version."
    "Jesus!!!!!!! Please Save Me!"
]

In [2]:
# Option 2: provide real timestamps
from datetime import datetime

real_timestamps = [
    datetime(2026, 4, 10, 9, 0, 5),    # 0: Figma idea
    datetime(2026, 4, 10, 9, 0, 42),   # 1: 
    datetime(2026, 4, 10, 9, 1, 15),   # 2: 
    datetime(2026, 4, 10, 9, 2, 30),   # 3:
    datetime(2026, 4, 10, 9, 15, 0),   # 4: 
    datetime(2026, 4, 10, 9, 15, 45),  # 5: 
    datetime(2026, 4, 10, 9, 20, 10),  # 6: 
    datetime(2026, 4, 10, 9, 21, 0),   # 7: 
    datetime(2026, 4, 10, 9, 25, 30),  # 8: 
    datetime(2026, 4, 10, 9, 26, 0),   # 9: 
    datetime(2026, 4, 10, 9, 27, 15),  # 10: 
    datetime(2026, 4, 10, 9, 28, 0),   # 11: 
    datetime(2026, 4, 10, 9, 29, 10),  # 12: 
    datetime(2026, 4, 10, 9, 45, 0),   # 13: 
    datetime(2026, 4, 10, 9, 46, 20),  # 14:
    datetime(2026, 4, 10, 9, 47, 0),   # 15: 
    datetime(2026, 4, 10, 9, 48, 30),  # 16: 
    datetime(2026, 4, 10, 9, 49, 15),  # 17: F..
    datetime(2026, 4, 10, 9, 55, 0),   # 18:
    datetime(2026, 4, 10, 9, 56, 10),  # 19: 
    datetime(2026, 4, 10, 9, 57, 30),  # 20: 
    datetime(2026, 4, 10, 10, 10, 0),  # 21:
    datetime(2026, 4, 10, 10, 11, 20), # 22: 
    datetime(2026, 4, 10, 10, 12, 0),  # 23: 
    datetime(2026, 4, 10, 10, 13, 15), # 24: 
    datetime(2026, 4, 10, 10, 20, 0),  # 25:
    datetime(2026, 4, 10, 10, 21, 30), # 26: F...
    datetime(2026, 4, 10, 10, 22, 45), # 27: 
    datetime(2026, 4, 10, 10, 30, 0),  # 28: 
    datetime(2026, 4, 10, 10, 45, 0),  # 29: 
    datetime(2026, 4, 10, 10, 46, 20), # 30: 
    datetime(2026, 4, 10, 10, 47, 0),  # 31: 
    datetime(2026, 4, 10, 10, 48, 30), # 32: 
    datetime(2026, 4, 10, 11, 0, 0),   # 33:
    datetime(2026, 4, 10, 11, 1, 15),  # 34: 
    datetime(2026, 4, 10, 11, 10, 0),  # 35: 
    datetime(2026, 4, 10, 11, 11, 0),  # 36: 
    datetime(2026, 4, 10, 11, 12, 30), # 37:
    datetime(2026, 4, 10, 11, 13, 45), # 38:
    datetime(2026, 4, 10, 11, 20, 0),  # 39: 
    datetime(2026, 4, 10, 11, 21, 10), # 40: 
    datetime(2026, 4, 10, 11, 22, 30), # 41: 
    datetime(2026, 4, 10, 11, 30, 0),  # 42: 
    datetime(2026, 4, 10, 11, 31, 15), # 43: 
    datetime(2026, 4, 10, 11, 45, 0),  # 44: 
    datetime(2026, 4, 10, 11, 46, 20), # 45: 
    datetime(2026, 4, 10, 12, 0, 0),   # 46: 
    datetime(2026, 4, 10, 12, 1, 15),  # 47: 
    datetime(2026, 4, 10, 12, 2, 30),  # 48: 
    datetime(2026, 4, 10, 12, 3, 45),  # 49:
    datetime(2026, 4, 10, 12, 30, 0),  # 50: Jesus! 
]


In [3]:
ranker = HarnessChandelier(
    weights={"delta_time": 0.2}
)

result = ranker.fit(messages, timestamps=real_timestamps)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
print(f"Main Topic: {result.main_topic}")
print()
print("=== PageRank (Topic Importance) ===")
print(result.pagerank)

Main Topic: 0

=== PageRank (Topic Importance) ===
    vertex  pagerank
0        0  0.185863
1        3  0.134400
2        6  0.105720
3        1  0.091297
4        8  0.085628
5        2  0.084069
6        7  0.075642
7        4  0.075631
8        5  0.075123
9        9  0.043335
10      -1  0.043292


In [5]:
from collections import Counter

print("=== Topic Distribution ===")
counter = Counter(result.topic_labels)
for topic, count in sorted(counter.items()):
    print(f"Topic {topic}: {count} count")

print()
print("=== Topic per Message ===")
for i, (msg, topic) in enumerate(zip(messages, result.topic_labels)):
    print(f"[{i:02d}] Topic {topic}: {msg[:55]}...")

=== Topic Distribution ===
Topic -1: 4 count
Topic 0: 9 count
Topic 1: 6 count
Topic 2: 5 count
Topic 3: 5 count
Topic 4: 5 count
Topic 5: 4 count
Topic 6: 4 count
Topic 7: 3 count
Topic 8: 3 count
Topic 9: 3 count

=== Topic per Message ===
[00] Topic -1: Hey, I have this idea for a website. I want to build so...
[01] Topic 1: The sidebar needs to be on the left side, and at the bo...
[02] Topic 0: So to clarify, it's definitely blue tones only. Not gre...
[03] Topic 5: When a person inputs a description, the site should aut...
[04] Topic 2: I just tried running the code and it threw an error. Wh...
[05] Topic 1: The input box is in the middle of the page right now. I...
[06] Topic 2: I ran the code again and it's still showing an error. W...
[07] Topic 4: Don't you remember? I said the website should be like F...
[08] Topic 3: Ah another error popped up again. Do I really have to r...
[09] Topic 0: It's blue! I said blue website! Not the whole page, jus...
[10] Topic -1: About file u

In [6]:
# Final Summary
print("=== Dominant Topic Analysis ===")
main_topic_messages = [
    (i, msg) for i, (msg, topic) 
    in enumerate(zip(messages, result.topic_labels)) 
    if topic == result.main_topic
]

print(f"Main Topic: {result.main_topic} (PageRank: {result.pagerank.iloc[0]['pagerank']:.4f})")
print(f"Appears in {len(main_topic_messages)} out of {len(messages)} messages")
print()
print("Messages classified as Main Topic:")
for i, msg in main_topic_messages:
    print(f"  [{i:02d}] {msg[:70]}...")

=== Dominant Topic Analysis ===
Main Topic: 0 (PageRank: 0.1859)
Appears in 9 out of 51 messages

Messages classified as Main Topic:
  [02] So to clarify, it's definitely blue tones only. Not green, not purple....
  [09] It's blue! I said blue website! Not the whole page, just the top part ...
  [12] The layout should be centered, with the main content area in white but...
  [16] Why did you change the blue color? I specifically said blue tones, not...
  [20] Can we go back to basics? Blue tones on top, white content area, sideb...
  [23] Stop changing things I didn't ask you to change. Just fix what I asked...
  [27] The blue color on top should be a gradient from dark blue to medium bl...
  [35] Now the blue styling broke on mobile. It looks completely different on...
  [38] The top section blue color - it should match the shade #1E3A8A specifi...
